# Running the Experiment

This experiment is implemented as a single runnable notebook block, allowing the entire training and validation pipeline to be executed with minimal manual intervention.

When the code is executed, the following steps are performed:

1. Library installation and import.  
The required libraries are installed at the beginning of the block, including transformers, datasets, accelerate, and scikit-learn. After installation, all necessary modules are imported, including PyTorch utilities, optimization tools, and evaluation metrics.

2. Device selection and seed initialization.  
The code automatically selects the computation device. If a CUDA-compatible GPU is available, execution is performed on GPU; otherwise, it falls back to CPU. Random seeds are then fixed for Python, NumPy, and PyTorch (including CUDA when available) to improve reproducibility across runs.

3. Tokenizer and pretrained model setup.  
The tokenizer associated with bert-base-uncased is loaded, and the model is initialized using AutoModelForSequenceClassification with two output labels. The architecture consists of a pretrained BERT encoder followed by a classification head. The model is then moved to the selected device.

4. Dataset loading.  
The SemEval-2026 Task 13 dataset (configuration A) is automatically downloaded and loaded using the Hugging Face datasets library. 

5. Tokenization and preprocessing.  
Each code sample is tokenized using the BERT tokenizer. Truncation and padding are applied to a maximum sequence length of 256 tokens. Labels are attached to each processed example, and unnecessary columns (such as raw code, generator, and language) are removed.

6. Formatting and batching.  
The processed datasets are converted into PyTorch tensor format and wrapped into DataLoader objects. The training set is shuffled to ensure randomness, while the validation set is processed in a fixed order. A batch size of 32 is used.

7. Initial freezing of the encoder.  
All parameters of the BERT encoder are initially frozen, meaning that they are not updated during backpropagation. This allows the model to start training in a stable configuration.

8. Gradual unfreezing strategy.  
Initially, the BERT encoder is frozen, after which the last four Transformer layers are unfrozen before training starts. At the beginning of each epoch, additional layers are unfrozen in groups of four. However, since the optimizer is not reinitialized after each unfreezing step, only the layers that were unfrozen at the time of optimizer creation are effectively updated during training.

9. Parameter grouping and learning rates.  
Model parameters are grouped into classifier parameters and non-classifier parameters, with only the trainable ones included in the optimizer at initialization. Different learning rates are assigned to each group, with a smaller learning rate for the encoder (2e-5) and a larger one (1e-4) for the classifier. The optimizer is initialized after the last four encoder layers are unfrozen, so it includes only those encoder layers (together with the classifier) in the optimization process.

10. Optimizer and scheduler setup.  
The optimizer (AdamW) is initialized using the defined parameter groups. A linear learning-rate scheduler with warmup is configured, with 6% of the total training steps used for warmup.

11. Evaluation function definition.  
A function is defined to evaluate the model on the validation set. The model is set to evaluation mode, predictions are computed using argmax over logits, and macro F1 is calculated.

12. Checkpoint configuration.  
Directories and file paths for saving model checkpoints are created. Both best-model saving and full checkpoint saving are supported.

13. Checkpoint loading (resume training).  
If a checkpoint exists, the model, optimizer, scheduler, best score, and epoch are restored. Otherwise, training starts from the first epoch.

14. Training loop initialization.  
The training process begins over the specified number of epochs. At the start of each epoch, additional encoder layers are unfrozen according to the progressive strategy.

15. Training mode activation.  
The model is set to training mode, and a list is initialized to track batch losses.

16. Batch processing.  
Each batch of training data is loaded and moved to the selected device, including input IDs, attention masks, and labels.

17. Forward pass and loss computation.  
The model processes the inputs and computes the loss, which is stored for monitoring purposes.

18. Backpropagation and gradient clipping.  
Gradients are computed via backpropagation, and gradient clipping is applied to maintain numerical stability.

19. Parameter updates and scheduler step.  
The optimizer updates the trainable parameters, the scheduler adjusts the learning rate, and gradients are reset.

20. Validation and metric computation.  
After each epoch, the model is evaluated on the validation set. Predictions are generated and the macro F1 score is computed.

21. Training statistics reporting.  
The average training loss and validation F1 score are printed for each epoch.

22. Best model saving.  
If the validation F1 score improves, the current model weights are saved as the best-performing model.

23. Checkpoint saving.  
A checkpoint containing the full training state is saved after each epoch to allow training to resume if interrupted.

24. Final model loading.  
After training, the best saved model is loaded and set to evaluation mode for further use.

## Runtime considerations

Running the experiment on CPU may significantly increase execution time. If memory limitations occur, the batch size can be reduced (for example, from 32 to 16 or 8). Evaluation on the validation set may also take noticeable time depending on hardware.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn
import os
import random
import numpy as np
import torch

from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

dispozitiv = torch.device("cuda" if torch.cuda.is_available() else "cpu")

samanta = 42
random.seed(samanta)
np.random.seed(samanta)
torch.manual_seed(samanta)
if dispozitiv.type == "cuda":
    torch.cuda.manual_seed_all(samanta)

nume_model = "bert-base-uncased"

tokenizator = AutoTokenizer.from_pretrained(nume_model)
model = AutoModelForSequenceClassification.from_pretrained(
    nume_model,
    num_labels=2
)
model.to(dispozitiv)

dataset_initial = load_dataset("DaniilOr/SemEval-2026-Task13", "A")

date_antrenare = dataset_initial["train"]
date_validare  = dataset_initial["validation"]
lungime_max = 256

def tokenizeaza(exemple):
    texte = exemple["code"]
    lbl   = exemple["label"]
    t = tokenizator(
        texte,
        truncation=True,
        padding="max_length",
        max_length=lungime_max
    )
    t["labels"] = lbl
    return t

train_proc = date_antrenare.map(
    tokenizeaza,
    batched=True,
    remove_columns=["code", "generator", "language"]
)

valid_proc = date_validare.map(
    tokenizeaza,
    batched=True,
    remove_columns=["code", "generator", "language"]
)
train_proc.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

valid_proc.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
marime_batch = 32

loader_train = DataLoader(
    train_proc,
    batch_size=marime_batch,
    shuffle=True
)

loader_valid = DataLoader(
    valid_proc,
    batch_size=marime_batch,
    shuffle=False
)
for p in model.bert.parameters():
    p.requires_grad = False

straturi = list(model.bert.encoder.layer)
total = len(straturi)
pas_deblocare = 4

def deblocheaza_straturi(n):
    n = max(0, min(total, n))
    start = total - n
    for i in range(total):
        strat = straturi[i]
        permite = (i >= start)
        for p in strat.parameters():
            p.requires_grad = permite

deblocheaza_straturi(pas_deblocare)
lista_enc = []
lista_clf = []
for nume, p in model.named_parameters():
    if "classifier" in nume:
        lista_clf.append(p)
    else:
        lista_enc.append(p)

lr_enc = 2e-5
lr_clf = 1e-4
nr_epoci = 3

opt = AdamW(
    [
        {"params": [x for x in lista_enc if x.requires_grad], "lr": lr_enc},
        {"params": [x for x in lista_clf if x.requires_grad], "lr": lr_clf},
    ]
)
pasi_epoca = len(loader_train)
total_pasi = pasi_epoca * nr_epoci
warm = int(total_pasi * 0.06)

scheduler = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=warm,
    num_training_steps=total_pasi
)

def testeaza():
    model.eval()
    toate_pred = []
    toate_adev = []

    with torch.no_grad():
        for lot in loader_valid:
            intrari = lot["input_ids"].to(dispozitiv)
            masca   = lot["attention_mask"].to(dispozitiv)
            etic    = lot["labels"].to(dispozitiv)

            ies = model(input_ids=intrari, attention_mask=masca)
            scoruri = ies.logits
            pred_lot = torch.argmax(scoruri, dim=1)

            toate_pred += pred_lot.cpu().numpy().tolist()
            toate_adev += etic.cpu().numpy().tolist()

    return f1_score(toate_adev, toate_pred, average="macro")

import os

os.makedirs("checkpoints", exist_ok=True)

fisier_model = "checkpoints/bert_finetuned_binar.pt"
cale_checkpoint = "checkpoints/checkpoint_bert_taskA.pt"

cel_mai_bun = 0.0
start_epoca = 0

if os.path.exists(cale_checkpoint):
    checkpoint = torch.load(cale_checkpoint, map_location=dispozitiv)
    model.load_state_dict(checkpoint["model_state"])
    opt.load_state_dict(checkpoint["opt_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    cel_mai_bun = checkpoint["cel_mai_bun"]
    start_epoca = checkpoint["epoca"] + 1
    print("Am gasit checkpoint. Reiau de la epoca:", start_epoca + 1)
else:
    print("Nu am gasit checkpoint. Pornesc de la epoca 1.")

for ep in range(start_epoca, nr_epoci):
    nr_deblocate = (ep + 1) * pas_deblocare
    deblocheaza_straturi(nr_deblocate)

    model.train()
    lista_pierderi = []

    for lot in loader_train:
        id_uri = lot["input_ids"].to(dispozitiv)
        masca  = lot["attention_mask"].to(dispozitiv)
        lab    = lot["labels"].to(dispozitiv)

        ies = model(
            input_ids=id_uri,
            attention_mask=masca,
            labels=lab
        )

        loss_curent = ies.loss
        lista_pierderi.append(loss_curent.item())

        loss_curent.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        opt.step()
        scheduler.step()
        opt.zero_grad()

    scor_f1 = testeaza()

    print(
        "Epoca:", ep + 1,
        "pierdere_medie:", float(np.mean(lista_pierderi)),
        "F1_macro_validare:", scor_f1
    )

    if scor_f1 > cel_mai_bun:
        cel_mai_bun = scor_f1
        torch.save(model.state_dict(), fisier_model)
        print("Model imbunatatit. F1_macro_validare:", cel_mai_bun)

    checkpoint = {
        "epoca": ep,
        "model_state": model.state_dict(),
        "opt_state": opt.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "cel_mai_bun": cel_mai_bun,
    }

    torch.save(checkpoint, cale_checkpoint)
    print("Checkpoint salvat dupa epoca", ep + 1)
state = torch.load(fisier_model, map_location=dispozitiv)
model.load_state_dict(state)
model.to(dispozitiv)
model.eval()


## Expected Output (Illustrative Example)

During execution, the training process prints progress information for each epoch.

An example output after the first epoch is shown below (messages are displayed in Romanian, as defined in the implementation):

```text
Nu am gasit checkpoint. Pornesc de la epoca 1.

Epoca: 1
pierdere_medie: ~0.15
F1_macro_validare: ~0.96

Model imbunatatit. F1_macro_validare: ~0.96
Checkpoint salvat dupa epoca 1


## Interpretation

The exact values may vary depending on the execution environment, random initialization, and whether training starts from scratch or from a saved checkpoint.

If the validation Macro F1 is higher than in the frozen-encoder baseline, this would suggest that partial fine-tuning helps the model adapt better to the target task.